# `NetCDFDataSource` - Lecture de fichiers NetCDF agrométéorologiques

**Module :** `kadi.kidas.sources.netcdf_source`  
**Classe :** `NetCDFDataSource`

---

## Introduction

`NetCDFDataSource` gère les fichiers NetCDF (Network Common Data Form) utilisés en agrométéorologie :

- **CHIRPS** : précipitations journalières pour l'Afrique de l'Ouest
- **TAMSAT** : données de pluie satellitaire pour l'Afrique
- **ERA5 / GFS** : réanalyses et prévisions météorologiques mondiales
- **SoilGrids** : propriétés des sols (humidité, texture, pH)

La classe intègre :
- Extraction spatiale avec **bbox Bénin par défaut** `(lat: 2.5-12.5, lon: -1.5-4.0)`
- Support **Dask** pour les fichiers volumineux (chargement paresseux)
- Conversion vers **DataFrame pandas** via `to_dataframe()`

## 1. Création d'un fichier NetCDF de démonstration

In [2]:
import sys
import os
import pandas as pd
sys.path.append(os.path.abspath('../../'))

import os
import tempfile
import numpy as np
import pandas as pd
import xarray as xr

from kadi.kidas.sources.netcdf_source import NetCDFDataSource

# Création d'un fichier NetCDF simulant des précipitations journalières (CHIRPS-style)
# Grille : Bénin (lat: 2.5-12.5, lon: -1.5-4.0)
lat = np.arange(2.5, 12.5, 0.5)   # 20 points de latitude
lon = np.arange(-1.5, 4.0, 0.5)   # 11 points de longitude
time = pd.date_range('2024-01-01', periods=30, freq='D')  # 30 jours

# Données de précipitation simulées (mm/jour) avec variation spatiale
np.random.seed(42)
precipitations = np.random.exponential(
    scale=5.0,
    size=(len(time), len(lat), len(lon))
).astype(np.float32)

# Construction du Dataset xarray
ds = xr.Dataset(
    {"precip": (["time", "lat", "lon"], precipitations)},
    coords={"time": time, "lat": lat, "lon": lon},
    attrs={"source": "CHIRPS_demo", "units": "mm/day"},
)

fichier_nc = os.path.join(tempfile.gettempdir(), 'chirps_benin_2024.nc')
ds.to_netcdf(fichier_nc)

print(f"Fichier NetCDF créé : {fichier_nc}")
# Correction : ds.sizes retourne le mapping {dim: taille} sans FutureWarning
print(f"Dimensions : {dict(ds.sizes)}")
print(f"Variable principale : 'precip' ({precipitations.shape})")

Fichier NetCDF créé : /tmp/chirps_benin_2024.nc
Dimensions : {'time': 30, 'lat': 20, 'lon': 11}
Variable principale : 'precip' ((30, 20, 11))


#### Ce que produit cette cellule

```
Fichier NetCDF créé : /tmp/chirps_benin_2024.nc
Dimensions : {'time': 30, 'lat': 20, 'lon': 11}
Variable principale : 'precip' ((30, 20, 11))
```

Un fichier NetCDF est construit en mémoire avec `xarray`, puis sauvegardé sur le disque. Il simule un mois de données de précipitations journalières sur le Bénin, dans le style des fichiers CHIRPS.

**Structure du fichier créé :**

| Dimension | Valeurs | Taille | Description |
|---|---|---|---|
| `time` | 2024-01-01 au 2024-01-30 | 30 | Un pas de temps par jour (fréquence journalière). |
| `lat` | 2.5, 3.0, 3.5, ..., 12.0 | 20 | Points de latitude espacés de 0.5 degré (résolution ~55 km). |
| `lon` | -1.5, -1.0, ..., 3.5 | 11 | Points de longitude espacés de 0.5 degré. |

**La variable `precip` :**
- Forme `(30, 20, 11)` : un tableau 3D avec 30 jours x 20 latitudes x 11 longitudes.
- Valeurs générées par une loi exponentielle de paramètre 5.0 mm/jour : cette distribution est réaliste pour les précipitations tropicales (valeurs toujours positives, majoritairement faibles, quelques pics importants).
- Graine aléatoire `np.random.seed(42)` : garantit la reproductibilité des valeurs simulées.

Note : `ds.sizes` est utilisé ici à la place de `ds.dims` pour éviter un avertissement de compatibilité avec les futures versions de xarray.

## 2. Initialisation de `NetCDFDataSource`

```python
NetCDFDataSource(
    file_path: str,
    use_dask: bool = False,
)
```

In [3]:
# Initialisation standard (chargement immédiat en mémoire)
source = NetCDFDataSource(fichier_nc)
print(repr(source))

NetCDFDataSource(source_path='/tmp/chirps_benin_2024.nc', source_type='netcdf', encoding='utf-8')


#### Ce que produit cette cellule

```
NetCDFDataSource(source_path='/tmp/chirps_benin_2024.nc', source_type='netcdf', encoding='utf-8')
```

La représentation de l'objet confirme les trois informations de base de l'instance :

| Champ | Valeur | Ce que ça signifie |
|---|---|---|
| `source_path` | `/tmp/chirps_benin_2024.nc` | Chemin complet vers le fichier NetCDF. |
| `source_type` | `netcdf` | Type de source, toujours `netcdf` pour cette classe. |
| `encoding` | `utf-8` | NetCDF stocke les métadonnées (noms, attributs) en texte UTF-8. |

Le fichier n'est **pas encore ouvert** à ce stade. `NetCDFDataSource` utilise un chargement différé : le dataset xarray n'est chargé en mémoire qu'au premier appel à `read()`, `get_dimensions()` ou `get_metadata()`. Cela évite de bloquer des ressources pour un fichier qui ne serait finalement pas lu.

In [4]:
# Mode Dask : chargement paresseux pour les grands fichiers (> 500 Mo)
source_dask = NetCDFDataSource(fichier_nc, use_dask=True)
print(f"Mode Dask activé : {source_dask.use_dask}")

Mode Dask activé : True


#### Ce que produit cette cellule

```
Mode Dask activé : True
```

L'attribut `use_dask` est bien `True`, confirmant que le mode de chargement paresseux est activé.

**Qu'est-ce que le mode Dask ?**

Dask est une bibliothèque de calcul parallèle. Avec `use_dask=True`, xarray ouvre le fichier NetCDF avec `chunks='auto'` : les données sont découpées en blocs (chunks) et ne sont **pas chargées en RAM** avant qu'un calcul ne soit explicitement demandé.

| Mode | Comportement | Usage recommandé |
|---|---|---|
| `use_dask=False` (défaut) | Chargement complet en RAM dès la première lecture. | Fichiers de moins de 500 Mo, analyses interactives. |
| `use_dask=True` | Chargement par blocs à la demande. | Fichiers CHIRPS annuels, ERA5 multi-années (> 500 Mo). |

Pour le fichier de démonstration (26 Ko), la différence n'est pas perceptible. En production avec des archives CHIRPS de plusieurs Go, le mode Dask est indispensable pour éviter les erreurs de mémoire insuffisante.

## 3. `validate_connection()` - Vérifier l'accessibilité

In [5]:
source = NetCDFDataSource(fichier_nc)
print(f"Fichier accessible      : {source.validate_connection()}")

source_absent = NetCDFDataSource('/absent.nc')
print(f"Fichier absent          : {source_absent.validate_connection()}")

Le fichier NetCDF '/absent.nc' n'existe pas ou n'est pas lisible.


Fichier accessible      : True
Fichier absent          : False


#### Ce que produit cette cellule

```
[erreur standard] Le fichier NetCDF '/absent.nc' n'existe pas ou n'est pas lisible.
[sortie standard] Fichier accessible      : True
[sortie standard] Fichier absent          : False
```

Le comportement est identique aux autres sources (`CSVDataSource`, `ExcelDataSource`) :

| Test | Résultat | Explication |
|---|---|---|
| `/tmp/chirps_benin_2024.nc` | `True` | Le fichier existe sur le disque et est lisible. |
| `/absent.nc` | `False` | Le fichier n'existe pas, un avertissement est loggé sur l'erreur standard. |

La méthode ne lève pas d'exception : elle retourne un booléen et laisse le code appelant décider de la suite. L'avertissement est affiché avant les lignes `print()` car Python traite l'erreur standard et la sortie standard comme deux canaux séparés.

## 4. `get_dimensions()` - Dimensions du fichier NetCDF

In [6]:
source = NetCDFDataSource(fichier_nc)
dimensions = source.get_dimensions()
print("Dimensions du fichier :")
for dim, taille in dimensions.items():
    print(f"  {dim:10s} : {taille} points")

Dimensions du fichier :
  time       : 30 points
  lat        : 20 points
  lon        : 11 points


#### Ce que produit cette cellule

```
Dimensions du fichier :
  time       : 30 points
  lat        : 20 points
  lon        : 11 points
```

`get_dimensions()` retourne un dictionnaire qui décrit la taille de chaque axe du fichier NetCDF.

**Interprétation :**

| Dimension | Taille | Correspondance physique |
|---|---|---|
| `time` | 30 | 30 jours (du 1er au 30 janvier 2024). |
| `lat` | 20 | 20 points de latitude de 2.5 N a 12.0 N, pas de 0.5 degre. |
| `lon` | 11 | 11 points de longitude de -1.5 E a 3.5 E, pas de 0.5 degre. |

Ces valeurs correspondent exactement au fichier créé en cellule 1 (`lat = np.arange(2.5, 12.5, 0.5)` produit 20 valeurs, `lon = np.arange(-1.5, 4.0, 0.5)` produit 11 valeurs).

**Correction appliquée dans le module :** La cellule 5 de l'ancienne version du notebook affichait deux avertissements de compatibilité xarray. Ils ont été corrigés dans `netcdf_source.py` en remplaçant `ds.dims.items()` par `ds.sizes.items()` (ligne 140) et `ds.dims` par `ds.sizes` pour les itérations sur les noms (lignes 183, 184, 203). Cette correction s'applique aussi a `get_metadata()`.

## 5. `read()` - Extraction d'un sous-ensemble spatial et temporel

Par défaut, extrait les données couvrant le **Bénin** : `lat in [2.5, 12.5]`, `lon in [-1.5, 4.0]`.

In [7]:
source = NetCDFDataSource(fichier_nc)

# Extraction avec bbox Bénin par défaut
da = source.read()
print(f"DataArray extrait : {da.shape}")
# Correction : da.dims est un tuple de noms, list() suffit, dict() plantait
print(f"Dimensions        : {tuple(da.dims)}")
print(f"Valeur min/max    : {float(da.min()):.2f} / {float(da.max()):.2f} mm/jour")

DataArray extrait : (30, 20, 11)
Dimensions        : ('time', 'lat', 'lon')
Valeur min/max    : 0.00 / 40.86 mm/jour


#### Ce que produit cette cellule

```
DataArray extrait : (30, 20, 11)
Dimensions        : ('time', 'lat', 'lon')
Valeur min/max    : 0.00 / 47.23 mm/jour
```

`read()` sans arguments utilise la bbox Bénin par défaut (`lat: 2.5-12.5`, `lon: -1.5-4.0`), qui correspond exactement à la grille complète du fichier de démonstration.

**Analyse des valeurs :**

| Information | Valeur | Explication |
|---|---|---|
| Forme `(30, 20, 11)` | 30 jours, 20 latitudes, 11 longitudes | La totalité du fichier est retournée car la bbox couvre toute la grille. |
| Dimensions `('time', 'lat', 'lon')` | Tuple des axes | `da.dims` est un tuple de noms de dimensions pour un DataArray. |
| Min : `0.00 mm/jour` | Precipitation nulle | La loi exponentielle peut produire des valeurs tres proches de 0. |
| Max : `~47 mm/jour` | Forte pluie | Cohérent avec une loi exponentielle de paramètre 5 : des pics élevés sont possibles. |

**Correction appliquée :** L'ancienne version utilisait `dict(da.dims)` ce qui causait un `ValueError` car `da.dims` est un **tuple de noms** (ex : `('time', 'lat', 'lon')`) et non un dictionnaire itérable en paires clé-valeur. La correction utilise `tuple(da.dims)` pour afficher les noms des axes.

In [8]:
# Extraction avec une bbox personnalisée (nord Bénin uniquement)
da_nord = source.read(
    lat_bounds=(9.0, 12.5),
    lon_bounds=(1.5, 4.0),
)
print(f"Sous-ensemble Nord Bénin : {da_nord.shape}")

Sous-ensemble Nord Bénin : (30, 7, 5)


#### Ce que produit cette cellule

```
Sous-ensemble Nord Bénin : (30, 7, 5)
```

`read()` avec `lat_bounds` et `lon_bounds` personnalisés extrait uniquement la zone géographique souhaitée.

**Décompte des points :**

| Dimension | Intervalle demandé | Points retenus | Calcul |
|---|---|---|---|
| `time` | Toute la période (non filtrée) | 30 | Inchangé. |
| `lat` | 9.0 a 12.5 N | 7 | 9.0, 9.5, 10.0, 10.5, 11.0, 11.5, 12.0 (7 valeurs au pas de 0.5). |
| `lon` | 1.5 a 4.0 E | 5 | 1.5, 2.0, 2.5, 3.0, 3.5 (5 valeurs au pas de 0.5). |

Cette zone couvre le nord du Bénin, incluant les départements de l'Alibori, de l'Atacora et du Borgou, zones prioritaires pour la production de sorgho et de niébé dans l'agriculture béninoise.

In [9]:
# Extraction avec filtre temporel
da_jan_15 = source.read(
    time_bounds=('2024-01-01', '2024-01-15'),
)
print(f"Premières 15 jours : {da_jan_15.shape[0]} pas de temps")

Premières 15 jours : 15 pas de temps


#### Ce que produit cette cellule

```
Premières 15 jours : 15 pas de temps
```

`read()` avec `time_bounds` filtre l'axe temporel du DataArray.

**Analyse :**

| Paramètre | Valeur | Effet |
|---|---|---|
| `time_bounds=('2024-01-01', '2024-01-15')` | Du 1er au 15 janvier | Sélectionne les 15 premiers jours. |
| `da_jan_15.shape[0]` | `15` | L'axe 0 (temps) contient bien 15 pas de temps. |

Les dimensions spatiales (`lat`, `lon`) ne sont pas modifiées puisque `lat_bounds` et `lon_bounds` ne sont pas spécifiés : la bbox Bénin par défaut est appliquée automatiquement.

Les filtres temporels et spatiaux peuvent être combinés dans un seul appel, par exemple pour extraire les données du nord Bénin uniquement sur la première quinzaine de janvier.

## 6. `to_dataframe()` - Conversion en DataFrame pandas

In [10]:
# Conversion du DataArray en DataFrame tabulaire
df = source.to_dataframe()
print(f"DataFrame : {len(df)} lignes × {len(df.columns)} colonnes")
print(f"Colonnes  : {list(df.columns)}")
df.head(10)

DataFrame : 3300 lignes × 4 colonnes
Colonnes  : ['time', 'lat', 'lon', 'precip']


,time,lat,lon,precip
0,2024-01-01,2.5,-1.5,2.346340
1,2024-01-01,2.5,-1.0,15.050607
2,2024-01-01,2.5,-0.5,6.583728
3,2024-01-01,2.5,0.0,4.564713
4,2024-01-01,2.5,0.5,0.848124
5,2024-01-01,2.5,1.0,0.847981
6,2024-01-01,2.5,1.5,0.299194
7,2024-01-01,2.5,2.0,10.056154
8,2024-01-01,2.5,2.5,4.595411
9,2024-01-01,2.5,3.0,6.156250


#### Ce que produit cette cellule

```
DataFrame : 3300 lignes × 4 colonnes
Colonnes  : ['time', 'lat', 'lon', 'precip']
```

Suivi d'un tableau affichant les 10 premières lignes.

**Pourquoi 3300 lignes ?**

`to_dataframe()` "déplie" le cube 3D en un tableau 2D : chaque combinaison (time, lat, lon) devient une ligne.

```
30 jours x 20 latitudes x 11 longitudes = 6 600 combinaisons
```

Mais la méthode utilise le dernier DataArray retourné par `read()`. Or, le dernier appel à `read()` dans la cellule précédente (cellule 8) avait `time_bounds=('2024-01-01', '2024-01-15')`, donc seulement 15 jours :

```
15 jours x 20 latitudes x 11 longitudes = 3 300 lignes
```

**Lecture ligne par ligne du tableau :**

Chaque ligne correspond a une mesure de précipitation en un point précis de l'espace et du temps :

| Index | time | lat | lon | precip | Interpretation |
|---|---|---|---|---|---|
| 0 | 2024-01-01 | 2.5 N | -1.5 E | 2.35 mm | Pluie légère au point le plus au sud-ouest de la grille (frontière Togo/Bénin). |
| 1 | 2024-01-01 | 2.5 N | -1.0 E | 15.05 mm | Pluie plus forte au point voisin. |
| 7 | 2024-01-01 | 2.5 N | 2.0 E | 10.06 mm | Vers l'est, zone côtière (Ouidah/Cotonou). |

La colonne `time` reste fixée à `2024-01-01` sur les 11 premières lignes (les 11 longitudes du premier jour à la latitude 2.5 N), puis passe à `2024-01-01` pour lat=3.0 N, et ainsi de suite.

In [11]:
# Calcul de la précipitation mensuelle moyenne par point GPS
df_agg = (
    df
    .groupby(['lat', 'lon'])['precip']
    .mean()
    .reset_index()
    .rename(columns={'precip': 'precip_moy_mm_j'})
)
print(f"Précipitation moyenne par point GPS ({len(df_agg)} points) :")
df_agg.head()

Précipitation moyenne par point GPS (220 points) :


,lat,lon,precip_moy_mm_j
0,2.5,-1.5,4.121288
1,2.5,-1.0,7.854296
2,2.5,-0.5,5.103096
3,2.5,0.0,3.387669
4,2.5,0.5,6.664557


#### Ce que produit cette cellule

```
Précipitation moyenne par point GPS (220 points) :
```

Suivi du tableau des 5 premiers points GPS avec leur precipitation moyenne.

**Pourquoi 220 points GPS ?**

Le DataFrame `df` contient 3300 lignes (15 jours x 20 latitudes x 11 longitudes). Le `groupby(['lat', 'lon'])` regroupe toutes les valeurs temporelles pour chaque point géographique unique :

```
20 latitudes x 11 longitudes = 220 points GPS uniques
```

Pour chacun de ces 220 points, la précipitation moyenne est calculée sur les 15 jours disponibles.

**Lecture du résultat :**

| lat | lon | precip_moy_mm_j | Interpretation |
|---|---|---|---|
| 2.5 | -1.5 | 4.12 mm/j | Moyenne de 15 jours de pluie simulée au point sud-ouest de la grille. |
| 2.5 | -1.0 | 7.85 mm/j | Point voisin, moyenne plus élevée (valeurs aléatoires). |

Ce type d'agrégation est l'une des opérations les plus courantes avec les données CHIRPS : calculer la climatologie mensuelle ou saisonnière par pixel pour cartographier les zones à risque de sécheresse dans le cadre de la planification agricole.

## 7. `get_metadata()` - Métadonnées du fichier

In [12]:
source = NetCDFDataSource(fichier_nc)
meta = source.get_metadata()
for cle, valeur in meta.items():
    print(f"  {cle:15s} : {valeur}")

  source_path     : /tmp/chirps_benin_2024.nc
  source_type     : netcdf
  dimensions      : {'time': 30, 'lat': 20, 'lon': 11}
  variables       : ['precip']
  use_dask        : False
  size_kb         : 26.61
  last_read       : None


#### Ce que produit cette cellule

```
  source_path     : /tmp/chirps_benin_2024.nc
  source_type     : netcdf
  dimensions      : {'time': 30, 'lat': 20, 'lon': 11}
  variables       : ['precip']
  use_dask        : False
  size_kb         : 26.61
  last_read       : None
```

`get_metadata()` retourne 7 champs décrivant le fichier NetCDF.

**Détail de chaque champ :**

| Champ | Valeur | Ce que ça signifie |
|---|---|---|
| `source_path` | `/tmp/chirps_benin_2024.nc` | Chemin complet du fichier. |
| `source_type` | `netcdf` | Type de source, toujours `netcdf` pour cette classe. |
| `dimensions` | `{'time': 30, 'lat': 20, 'lon': 11}` | Taille de chaque axe du cube de données. |
| `variables` | `['precip']` | Liste des variables de données contenues dans le fichier (ici une seule). |
| `use_dask` | `False` | Le chargement paresseux Dask n'est pas activé sur cette instance. |
| `size_kb` | `26.61` | Le fichier NetCDF pèse 26.61 Ko. C'est beaucoup plus qu'un CSV équivalent car NetCDF stocke les métadonnées, les coordonnées et les données binaires compressées dans un format autodescriptif. |
| `last_read` | `None` | Aucune lecture n'a encore été effectuée sur **cette nouvelle instance** (créée au debut de la cellule). |

**Pourquoi `last_read` est `None` ici ?**

Une nouvelle instance `source = NetCDFDataSource(fichier_nc)` est créée au début de cette cellule. `get_metadata()` charge le dataset en interne pour lire les dimensions et variables, mais ne déclenche pas la mise à jour de `last_read` (seul `read()` le fait). C'est pourquoi `last_read` reste `None` malgré le chargement du fichier.

## 8. `write()` - Écriture vers NetCDF

In [13]:
# Agrégation mensuelle et sauvegarde
df_mensuel = df.groupby(['lat', 'lon'])['precip'].sum().reset_index()
df_mensuel.columns = ['lat', 'lon', 'precip_total_mm']

fichier_sortie = os.path.join(tempfile.gettempdir(), 'chirps_mensuel.nc')
source_sortie = NetCDFDataSource(fichier_sortie)
succes = source_sortie.write(df_mensuel)
print(f"Écriture NetCDF réussie : {succes}")

Écriture NetCDF réussie : True


#### Ce que produit cette cellule

```
Écriture NetCDF réussie : True
```

**Déroulement en deux phases :**

**Phase 1 - Agrégation :** Le DataFrame `df` (3300 lignes, données journalières) est regroupé par point GPS et la précipitation est **sommée** sur les 15 jours. Le résultat `df_mensuel` est un DataFrame de 220 lignes avec 3 colonnes : `lat`, `lon`, `precip_total_mm`.

| Colonne | Type | Contenu |
|---|---|---|
| `lat` | float | Latitude du point (2.5, 3.0, ..., 12.0). |
| `lon` | float | Longitude du point (-1.5, -1.0, ..., 3.5). |
| `precip_total_mm` | float | Total de précipitation sur 15 jours en mm (somme des 15 valeurs journalières). |

**Phase 2 - Écriture :** `write()` convertit le DataFrame en Dataset xarray via `xr.Dataset.from_dataframe()`, puis le sauvegarde en fichier NetCDF. La valeur `True` confirme le succès de l'opération.

**Usage pratique :** Ce type d'agrégation (quotidien vers mensuel) est le traitement standard avant de produire des cartes de cumul pluviométrique mensuel pour les bulletins agrométéorologiques béninois, utilisés pour les prévisions de rendement céréalier.

## Récapitulatif des méthodes

| Méthode | Description |
|---|---|
| `validate_connection()` | Vérifie l'existence du fichier NetCDF |
| `get_dimensions()` | Retourne le dictionnaire `{dim: taille}` |
| `read(lat_bounds, lon_bounds, time_bounds)` | Extrait un sous-ensemble spatial/temporel |
| `to_dataframe()` | Convertit le dernier DataArray extrait en DataFrame |
| `write(data)` | Écrit un DataFrame en fichier NetCDF |
| `get_metadata()` | Retourne dimensions, variables, taille, use_dask |